# {mod}`ultralytics` 训练模式

训练模式：在自定义或预载数据集上对模型进行微调。

In [1]:
import sys
_path = "/media/pc/data/lxw/ai/ultralytics" # ultralytics api 所在目录
sys.path.append(_path)
from pathlib import Path
from ultralytics import settings
temp_dir = Path("../.temp") # 设置缓存目录
temp_dir.mkdir(exist_ok=True, parents=True)
# 更新项目配置
settings.update({'weights_dir': f'{temp_dir}/weights'})

训练深度学习模型包括向其输入数据并调整参数，以便能够进行准确的预测。Ultralytics YOLOv11 中的训练模式专为高效和有效地训练目标检测模型而设计，充分利用了现代硬件的能力。本指南旨在涵盖您开始使用 YOLOv11 的全面功能集训练自己的模型所需的所有详细信息。

```{admonition} 为什么选择 Ultralytics YOLO 进行模型训练？
- 效率：充分利用硬件，无论您是使用单个 GPU 设置还是在多个 GPU 之间扩展。
- 多功能性：除 COCO、VOC 和 ImageNet 等现成数据集外，还可在自定义数据集上进行训练。
- 用户友好型：简单而强大的 CLI 和 Python 界面，提供直接的培训体验。
- 超参数灵活性：可定制的超参数范围广泛，可对模型性能进行微调。
```

```{admonition} 训练模式的主要功能
- 自动下载数据集：首次使用时会自动下载 COCO、VOC 和 ImageNet 等标准数据集。
支持多个 GPU ：在多个 GPU 上无缝扩展您的培训工作，以加快进程。
- 超参数配置：通过 YAML 配置文件或CLI 参数修改超参数的选项。
- 可视化和监控：实时跟踪培训指标和可视化学习过程，以获得更好的洞察力。
```

在 COCO8 数据集上，使用 640 的图像大小对 YOLOv11n 进行 100 个 epoch 的训练。可以通过 `device` 参数指定训练设备。如果没有传递参数且 GPU可用，则 `device=0` 将被使用，否则 `device='cpu'` 将被使用。

```python
from ultralytics import YOLO
# 加载模型
model = YOLO("yolo11n.yaml")  # 利用 YAML 构建新模型
model = YOLO("yolo11n.pt")  # 加载预训练模型（推荐用于训练）
model = YOLO("yolo11n.yaml").load("yolo11n.pt") # 从 YAML 构建并迁移权重
# 训练模型
results = model.train(data="coco8.yaml", epochs=100, imgsz=640)
```

## 多 GPU 训练

通过将训练负载分配到多个 GPU 上，多GPU训练能够更有效地利用可用的硬件资源。这一特性既可以通过 Python API 也可以通过命令行界面来实现。为了启用多 GPU 训练，您需要指定希望使用的 GPU 设备 ID。

要使用 2 个 GPU（CUDA 设备 0 和 1）进行训练，请使用以下命令。根据需要扩展到其他 GPU。

```python
from ultralytics import YOLO
# 加载模型
model = YOLO("yolo11n.pt")  # 加载预训练模型（建议用于训练）
# 使用 2 个 GPU 训练
results = model.train(data="coco8.yaml", epochs=100, imgsz=640, device=[0, 1])
```

## 苹果硅 MPS 训练

随着对 Ultralytics YOLO 模型中集成的苹果硅芯片的支持，现在可以在利用强大的 Metal Performance Shaders (MPS) 框架的设备上训练您的模型。MPS 为在苹果定制硅上执行计算和图像处理任务提供了高性能的方式。

为了在苹果硅芯片上启用训练，您应该在启动训练过程时指定 `'mps'` 作为您的设备。以下是如何在 Python 中以及通过命令行进行操作的示例：
```python
from ultralytics import YOLO

# Load a model
model = YOLO("yolo11n.pt")  # load a pretrained model (recommended for training)

# Train the model with MPS
results = model.train(data="coco8.yaml", epochs=100, imgsz=640, device="mps")
```

利用苹果硅芯片的计算能力，可以更高效地处理训练任务。如需更详细的指导和高级配置选项，请参阅 [PyTorch MPS 文档](https://pytorch.org/docs/stable/notes/mps.html)。

## 恢复中断的训练

在处理深度学习模型时，从之前保存的状态恢复训练其关键特性。这在多种情况下都非常有用，例如当训练过程意外中断，或者您希望使用新数据或更多周期继续训练模型时。

当训练恢复时，Ultralytics YOLO 会加载最后一次保存的模型权重，并恢复优化器状态、学习率调度器和周期数。这样，您就可以无缝地从停止的地方继续训练过程。

在 Ultralytics YOLO 中，您可以通过在调用 `train` 方法时将 `resume` 参数设置为 `True`，并指定包含部分训练模型权重的 `.pt` 文件路径来轻松恢复训练。

下面是使用 Python 以及通过命令行恢复中断训练的示例：

```python 
from ultralytics import YOLO
# Load a model
model = YOLO("path/to/last.pt")  # load a partially trained model
# Resume training
results = model.train(resume=True)
```

通过设置 `resume=True`，`train` 函数将从中断的地方继续训练，利用存储在 `'path/to/last.pt'` 文件中的状态。如果省略 `resume` 参数或将其设置为 `False`，则训练函数将从零开始训练。

请牢记，默认情况下检查点是在每个时期的末尾保存的，或者使用 `save_period` 参数在固定间隔进行保存。因此，您至少需要完成一个 epoch 才能恢复训练运行。

## 训练设置

YOLO 模型的训练设置包括在训练过程中使用的各种超参数和配置。这些设置影响模型的性能、速度和准确性（参见链接）。关键的训练设置包括批量大小、学习率、动量和权重衰减。此外，优化器的选择、损失函数以及训练数据集的构成也会影响训练过程。对这些设置进行仔细调整和实验对于优化性能至关重要。

| Argument          | Default  | Description                                                                                                                                                                                                                                                  |
| ----------------- | -------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| `model`           | `None`   | Specifies the model file for training. Accepts a path to either a `.pt` pretrained model or a `.yaml` configuration file. Essential for defining the model structure or initializing weights.                                                                |
| `data`            | `None`   | Path to the dataset configuration file (e.g., `coco8.yaml`). This file contains dataset-specific parameters, including paths to training and [validation data](https://www.ultralytics.com/glossary/validation-data), class names, and number of classes.    |
| `epochs`          | `100`    | Total number of training epochs. Each [epoch](https://www.ultralytics.com/glossary/epoch) represents a full pass over the entire dataset. Adjusting this value can affect training duration and model performance.                                           |
| `time`            | `None`   | Maximum training time in hours. If set, this overrides the `epochs` argument, allowing training to automatically stop after the specified duration. Useful for time-constrained training scenarios.                                                          |
| `patience`        | `100`    | Number of epochs to wait without improvement in validation metrics before early stopping the training. Helps prevent [overfitting](https://www.ultralytics.com/glossary/overfitting) by stopping training when performance plateaus.                         |
| `batch`           | `16`     | [Batch size](https://www.ultralytics.com/glossary/batch-size), with three modes: set as an integer (e.g., `batch=16`), auto mode for 60% GPU memory utilization (`batch=-1`), or auto mode with specified utilization fraction (`batch=0.70`).               |
| `imgsz`           | `640`    | Target image size for training. All images are resized to this dimension before being fed into the model. Affects model [accuracy](https://www.ultralytics.com/glossary/accuracy) and computational complexity.                                              |
| `save`            | `True`   | Enables saving of training checkpoints and final model weights. Useful for resuming training or [model deployment](https://www.ultralytics.com/glossary/model-deployment).                                                                                   |
| `save_period`     | `-1`     | Frequency of saving model checkpoints, specified in epochs. A value of -1 disables this feature. Useful for saving interim models during long training sessions.                                                                                             |
| `cache`           | `False`  | Enables caching of dataset images in memory (`True`/`ram`), on disk (`disk`), or disables it (`False`). Improves training speed by reducing disk I/O at the cost of increased memory usage.                                                                  |
| `device`          | `None`   | Specifies the computational device(s) for training: a single GPU (`device=0`), multiple GPUs (`device=0,1`), CPU (`device=cpu`), or MPS for Apple silicon (`device=mps`).                                                                                    |
| `workers`         | `8`      | Number of worker threads for data loading (per `RANK` if Multi-GPU training). Influences the speed of data preprocessing and feeding into the model, especially useful in multi-GPU setups.                                                                  |
| `project`         | `None`   | Name of the project directory where training outputs are saved. Allows for organized storage of different experiments.                                                                                                                                       |
| `name`            | `None`   | Name of the training run. Used for creating a subdirectory within the project folder, where training logs and outputs are stored.                                                                                                                            |
| `exist_ok`        | `False`  | If True, allows overwriting of an existing project/name directory. Useful for iterative experimentation without needing to manually clear previous outputs.                                                                                                  |
| `pretrained`      | `True`   | Determines whether to start training from a pretrained model. Can be a boolean value or a string path to a specific model from which to load weights. Enhances training efficiency and model performance.                                                    |
| `optimizer`       | `'auto'` | Choice of optimizer for training. Options include `SGD`, `Adam`, `AdamW`, `NAdam`, `RAdam`, `RMSProp` etc., or `auto` for automatic selection based on model configuration. Affects convergence speed and stability.                                         |
| `verbose`         | `False`  | Enables verbose output during training, providing detailed logs and progress updates. Useful for debugging and closely monitoring the training process.                                                                                                      |
| `seed`            | `0`      | Sets the random seed for training, ensuring reproducibility of results across runs with the same configurations.                                                                                                                                             |
| `deterministic`   | `True`   | Forces deterministic algorithm use, ensuring reproducibility but may affect performance and speed due to the restriction on non-deterministic algorithms.                                                                                                    |
| `single_cls`      | `False`  | Treats all classes in multi-class datasets as a single class during training. Useful for binary classification tasks or when focusing on object presence rather than classification.                                                                         |
| `rect`            | `False`  | Enables rectangular training, optimizing batch composition for minimal padding. Can improve efficiency and speed but may affect model accuracy.                                                                                                              |
| `cos_lr`          | `False`  | Utilizes a cosine [learning rate](https://www.ultralytics.com/glossary/learning-rate) scheduler, adjusting the learning rate following a cosine curve over epochs. Helps in managing learning rate for better convergence.                                   |
| `close_mosaic`    | `10`     | Disables mosaic [data augmentation](https://www.ultralytics.com/glossary/data-augmentation) in the last N epochs to stabilize training before completion. Setting to 0 disables this feature.                                                                |
| `resume`          | `False`  | Resumes training from the last saved checkpoint. Automatically loads model weights, optimizer state, and epoch count, continuing training seamlessly.                                                                                                        |
| `amp`             | `True`   | Enables Automatic [Mixed Precision](https://www.ultralytics.com/glossary/mixed-precision) (AMP) training, reducing memory usage and possibly speeding up training with minimal impact on accuracy.                                                           |
| `fraction`        | `1.0`    | Specifies the fraction of the dataset to use for training. Allows for training on a subset of the full dataset, useful for experiments or when resources are limited.                                                                                        |
| `profile`         | `False`  | Enables profiling of ONNX and TensorRT speeds during training, useful for optimizing model deployment.                                                                                                                                                       |
| `freeze`          | `None`   | Freezes the first N layers of the model or specified layers by index, reducing the number of trainable parameters. Useful for fine-tuning or [transfer learning](https://www.ultralytics.com/glossary/transfer-learning).                                    |
| `lr0`             | `0.01`   | Initial learning rate (i.e. `SGD=1E-2`, `Adam=1E-3`) . Adjusting this value is crucial for the optimization process, influencing how rapidly model weights are updated.                                                                                      |
| `lrf`             | `0.01`   | Final learning rate as a fraction of the initial rate = (`lr0 * lrf`), used in conjunction with schedulers to adjust the learning rate over time.                                                                                                            |
| `momentum`        | `0.937`  | Momentum factor for SGD or beta1 for [Adam optimizers](https://www.ultralytics.com/glossary/adam-optimizer), influencing the incorporation of past gradients in the current update.                                                                          |
| `weight_decay`    | `0.0005` | L2 [regularization](https://www.ultralytics.com/glossary/regularization) term, penalizing large weights to prevent overfitting.                                                                                                                              |
| `warmup_epochs`   | `3.0`    | Number of epochs for learning rate warmup, gradually increasing the learning rate from a low value to the initial learning rate to stabilize training early on.                                                                                              |
| `warmup_momentum` | `0.8`    | Initial momentum for warmup phase, gradually adjusting to the set momentum over the warmup period.                                                                                                                                                           |
| `warmup_bias_lr`  | `0.1`    | Learning rate for bias parameters during the warmup phase, helping stabilize model training in the initial epochs.                                                                                                                                           |
| `box`             | `7.5`    | Weight of the box loss component in the [loss function](https://www.ultralytics.com/glossary/loss-function), influencing how much emphasis is placed on accurately predicting [bounding box](https://www.ultralytics.com/glossary/bounding-box) coordinates. |
| `cls`             | `0.5`    | Weight of the classification loss in the total loss function, affecting the importance of correct class prediction relative to other components.                                                                                                             |
| `dfl`             | `1.5`    | Weight of the distribution focal loss, used in certain YOLO versions for fine-grained classification.                                                                                                                                                        |
| `pose`            | `12.0`   | Weight of the pose loss in models trained for pose estimation, influencing the emphasis on accurately predicting pose keypoints.                                                                                                                             |
| `kobj`            | `2.0`    | Weight of the keypoint objectness loss in pose estimation models, balancing detection confidence with pose accuracy.                                                                                                                                         |
| `label_smoothing` | `0.0`    | Applies label smoothing, softening hard labels to a mix of the target label and a uniform distribution over labels, can improve generalization.                                                                                                              |
| `nbs`             | `64`     | Nominal batch size for normalization of loss.                                                                                                                                                                                                                |
| `overlap_mask`    | `True`   | Determines whether object masks should be merged into a single mask for training, or kept separate for each object. In case of overlap, the smaller mask is overlayed on top of the larger mask during merge.                                                |
| `mask_ratio`      | `4`      | Downsample ratio for segmentation masks, affecting the resolution of masks used during training.                                                                                                                                                             |
| `dropout`         | `0.0`    | Dropout rate for regularization in classification tasks, preventing overfitting by randomly omitting units during training.                                                                                                                                  |
| `val`             | `True`   | Enables validation during training, allowing for periodic evaluation of model performance on a separate dataset.                                                                                                                                             |
| `plots`           | `False`  | Generates and saves plots of training and validation metrics, as well as prediction examples, providing visual insights into model performance and learning progression.                                                                                     |


```{admonition} 批处理大小设置说明

批处理参数可通过以下三种方式配置：

- 固定[批处理大小](https://www.ultralytics.com/glossary/batch-size)：设置一个整数值（例如，`batch=16`），直接指定每批的图像数量。
- 自动模式（$60\%$ GPU 内存）：使用 `batch=-1` 以自动调整批处理大小，大约利用 $60\%$ 的 CUDA 内存。
- 带利用率分数的自动模式：设置分数值（例如，`batch=0.70`），根据指定的 GPU 内存使用比例来调整批处理大小。
```

## 增强设置和超参数

增强技术对于提升 YOLO 模型的鲁棒性和性能至关重要，它通过向训练数据引入变异性，帮助模型更好地泛化到未见数据。下表概述了每个增强参数的目的和效果：

| Argument          | Type    | Default       | Range         | Description                                                                                                                                                               |
| ----------------- | ------- | ------------- | ------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| `hsv_h`           | `float` | `0.015`       | `0.0 - 1.0`   | Adjusts the hue of the image by a fraction of the color wheel, introducing color variability. Helps the model generalize across different lighting conditions.            |
| `hsv_s`           | `float` | `0.7`         | `0.0 - 1.0`   | Alters the saturation of the image by a fraction, affecting the intensity of colors. Useful for simulating different environmental conditions.                            |
| `hsv_v`           | `float` | `0.4`         | `0.0 - 1.0`   | Modifies the value (brightness) of the image by a fraction, helping the model to perform well under various lighting conditions.                                          |
| `degrees`         | `float` | `0.0`         | `-180 - +180` | Rotates the image randomly within the specified degree range, improving the model's ability to recognize objects at various orientations.                                 |
| `translate`       | `float` | `0.1`         | `0.0 - 1.0`   | Translates the image horizontally and vertically by a fraction of the image size, aiding in learning to detect partially visible objects.                                 |
| `scale`           | `float` | `0.5`         | `>=0.0`       | Scales the image by a gain factor, simulating objects at different distances from the camera.                                                                             |
| `shear`           | `float` | `0.0`         | `-180 - +180` | Shears the image by a specified degree, mimicking the effect of objects being viewed from different angles.                                                               |
| `perspective`     | `float` | `0.0`         | `0.0 - 0.001` | Applies a random perspective transformation to the image, enhancing the model's ability to understand objects in 3D space.                                                |
| `flipud`          | `float` | `0.0`         | `0.0 - 1.0`   | Flips the image upside down with the specified probability, increasing the data variability without affecting the object's characteristics.                               |
| `fliplr`          | `float` | `0.5`         | `0.0 - 1.0`   | Flips the image left to right with the specified probability, useful for learning symmetrical objects and increasing dataset diversity.                                   |
| `bgr`             | `float` | `0.0`         | `0.0 - 1.0`   | Flips the image channels from RGB to BGR with the specified probability, useful for increasing robustness to incorrect channel ordering.                                  |
| `mosaic`          | `float` | `1.0`         | `0.0 - 1.0`   | Combines four training images into one, simulating different scene compositions and object interactions. Highly effective for complex scene understanding.                |
| `mixup`           | `float` | `0.0`         | `0.0 - 1.0`   | Blends two images and their labels, creating a composite image. Enhances the model's ability to generalize by introducing label noise and visual variability.             |
| `copy_paste`      | `float` | `0.0`         | `0.0 - 1.0`   | Copies objects from one image and pastes them onto another, useful for increasing object instances and learning object occlusion.                                         |
| `copy_paste_mode` | `str`   | `flip`        | -             | Copy-Paste augmentation method selection among the options of (`"flip"`, `"mixup"`).                                                                                      |
| `auto_augment`    | `str`   | `randaugment` | -             | Automatically applies a predefined augmentation policy (`randaugment`, `autoaugment`, `augmix`), optimizing for classification tasks by diversifying the visual features. |
| `erasing`         | `float` | `0.4`         | `0.0 - 0.9`   | Randomly erases a portion of the image during classification training, encouraging the model to focus on less obvious features for recognition.                           |
| `crop_fraction`   | `float` | `1.0`         | `0.1 - 1.0`   | Crops the classification image to a fraction of its size to emphasize central features and adapt to object scales, reducing background distractions.                      |


这些设置可以根据数据集和当前任务的具体要求进行调整。尝试不同的数值有助于找到最优的数据增强策略，从而提升模型的性能表现。

## 训练日志

在训练 YOLOv11 模型时，跟踪模型随时间的性能变化是非常有价值的。这时就需要用到日志记录功能。Ultralytics 的 YOLO 支持三种类型的日志记录器 - Comet、ClearML 和 TensorBoard。

### Comet

[Comet](https://docs.ultralytics.com/integrations/comet/#comet-ml)平台，它允许数据科学家和开发人员追踪、比较、解释和优化实验及模型。它提供了实时指标、代码差异对比以及超参数跟踪等功能。

```python
# pip install comet_ml
import comet_ml

comet_ml.init()
```

请记得在 Comet 网站的个人账户中登录，并获取您的 API 密钥。您需要将此密钥添加到环境变量或脚本中，以便记录您的实验数据。

### ClearML

[ClearML](https://clear.ml/) 开源平台，它自动化跟踪实验并帮助高效共享资源。该平台旨在帮助团队更有效地管理、执行和复现他们的机器学习工作。

```python
# pip install clearml
import clearml

clearml.browser_login()
```

执行此脚本后，您需要在浏览器上登录您的 ClearML 账户并验证会话。

### TensorBoard

[TensorBoard](https://www.tensorflow.org/tensorboard)是用于[TensorFlow](https://www.ultralytics.com/glossary/tensorflow)的可视化工具包。它允许你可视化你的TensorFlow图，绘制关于图执行的定量指标图表，并显示通过它的额外数据，如图像。

要在[Google Colab](https://colab.research.google.com/github/ultralytics/ultralytics/blob/main/examples/tutorial.ipynb)中使用TensorBoard：
```bash
load_ext tensorboard
tensorboard --logdir ultralytics/runs  # replace with 'runs' directory
```

要本地运行 TensorBoard，请执行以下命令并在 <http://localhost:6006/> 查看结果。

```bash
tensorboard --logdir ultralytics/runs  # replace with 'runs' directory
```

这将启动 TensorBoard，并将其指向您保存训练日志的目录。

配置好记录器后，您可以继续进行模型训练。所有训练指标都将自动记录在您选择的平台中，您可以访问这些日志来监控模型随时间的性能变化，比较不同模型，并识别改进的空间。